# پروژه تحلیل و پیاده‌سازی مدل‌های یادگیری ماشین
---
در این دفترچه، فرآیند بارگذاری داده‌ها، پیش‌پردازش، پیاده‌سازی مدل **رگرسیون خطی چندگانه (Multiple Linear Regression)** و نحوه بارگذاری و استفاده از مدل **Random Forest Classifier** موجود در پوشه `models` پیاده‌سازی شده است.

## ۱. فراخوانی کتابخانه‌های مورد نیاز

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import mean_squared_error, r2_score, classification_report, accuracy_score

# تنظیمات مربوط به نمودارها
plt.style.use('ggplot')
import warnings
warnings.filterwarnings('ignore')

## ۲. بارگذاری و آماده‌سازی داده‌ها
> **نکته:** لطفاً مسیر فایل خود را در متغیر `dataset_path` قرار دهید.

In [ ]:
# فرض بر این است که دیتاست شما شامل ویژگی‌های ارسالی در تصویر است
dataset_path = "protein_dataset.csv" 

if os.path.exists(dataset_path):
    df = pd.read_csv(dataset_path)
    print("دیتاست با موفقیت بارگذاری شد.")
    display(df.head())
else:
    print(f"خطا: فایلی در مسیر {dataset_path} یافت نشد. در حال تولید داده‌های فرضی برای اجرای کد...")
    # تولید داده‌های فرضی بر اساس ساختار مدل‌های شما
    np.random.seed(42)
    n_samples = 500
    df = pd.DataFrame({
        'Age': np.random.randint(18, 60, n_samples),
        'Is_Male': np.random.choice([0, 1], n_samples),
        'Gender_Encoded': np.random.choice([0, 1], n_samples), # برای مدل رندوم فارست
        'Weight_kg': np.random.normal(70, 15, n_samples),
        'Height_cm': np.random.normal(170, 10, n_samples),
        'BMI': np.random.normal(24, 4, n_samples),
        'Body_Fat_Percent': np.random.normal(20, 5, n_samples),
        'Lean_Mass_kg': np.random.normal(55, 10, n_samples),
        'Activity_Score': np.random.uniform(1, 5, n_samples),
        'Daily_Protein_Intake_g': np.random.normal(120, 30, n_samples),
        'Genetic_Score': np.random.uniform(50, 100, n_samples),
        'Protein_Class': np.random.choice(['Casein', 'Plant_Protein', 'Whey_Concentrate', 'Whey_Isolate'], n_samples)
    })

## ۳. بخش اول: رگرسیون خطی چندگانه (Multiple Linear Regression)
در این بخش به پیش‌بینی یک متغیر پیوسته (مانند `Daily_Protein_Intake_g`) می‌پردازیم.

In [ ]:
# تعیین ویژگی‌ها (Features) و متغیر هدف (Target)
linear_features = ['Age', 'Is_Male', 'Weight_kg', 'Height_cm', 'BMI', 'Body_Fat_Percent', 'Lean_Mass_kg', 'Activity_Score', 'Genetic_Score']
target_linear = 'Daily_Protein_Intake_g'

X_reg = df[linear_features]
y_reg = df[target_linear]

# تقسیم داده‌ها به مجموعه‌های آموزش و تست
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

# ساخت و آموزش مدل
mlR_model = LinearRegression()
mlR_model.fit(X_train_reg, y_train_reg)

# پیش‌بینی روی داده‌های تست
y_pred_reg = mlR_model.predict(X_test_reg)

# ارزیابی مدل رگرسیون
mse = mean_squared_error(y_test_reg, y_pred_reg)
r2 = r2_score(y_test_reg, y_pred_reg)

print("=== نتایج ارزیابی رگرسیون خطی چندگانه ===")
print(f"Mean Squared Error (MSE): {mse:.4f}")
print(f"R2 Score: {r2:.4f}")

### ۳.۱. تصویرسازی نتایج رگرسیون

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(y_test_reg, y_pred_reg, color='blue', alpha=0.6, edgecolors='k')
plt.plot([y_test_reg.min(), y_test_reg.max()], [y_test_reg.min(), y_test_reg.max()], 'r--', lw=2)
plt.xlabel('Actual Values')
plt.ylabel('Predicted Values')
plt.title('Actual vs Predicted (Linear Regression)')
plt.show()

## ۴. بخش دوم: بارگذاری و استفاده از مدل‌های ذخیره شده (`RandomForestClassifier`)
با توجه به محتویات پوشه `models` شما، یک مدل طبقه‌بندی جنگل تصادفی برای دسته‌بندی نوع پروتئین مصرفی وجود دارد. در این بخش آن را بارگذاری می‌کنیم.

In [ ]:
# تعریف مسیر پوشه مدل‌ها
models_dir = "models"
model_name = "random_forest_model.pkl" # نام فایل مدل خود را اینجا دقیق وارد کنید
model_path = os.path.join(models_dir, model_name)

# ساخت پوشه فرضی در صورت عدم وجود جهت جلوگیری از ارور
if not os.path.exists(models_dir):
    os.makedirs(models_dir)

if os.path.exists(model_path):
    # بارگذاری مدل واقعی شما
    clf_model = joblib.load(model_path)
    print(f"مدل {model_name} با موفقیت بارگذاری شد.")
else:
    print(f"مدل واقعی در مسیر {model_path} یافت نشد. یک مدل جدید به عنوان نمونه ساخته و ذخیره می‌شود...")
    
    # ویژگی‌های مشخص شده در فایل متنی شما برای مدل رندوم فارست
    rf_features = ['Age', 'Gender_Encoded', 'Weight_kg', 'Height_cm', 'BMI', 'Body_Fat_Percent', 'Lean_Mass_kg', 'Activity_Score', 'Genetic_Score']
    X_clf = df[rf_features]
    y_clf = df['Protein_Class']
    
    X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(X_clf, y_clf, test_size=0.2, random_state=42)
    
    clf_model = RandomForestClassifier(n_estimators=100, random_state=42)
    clf_model.fit(X_train_clf, y_train_clf)
    
    # ذخیره مدل برای دفعات بعدی
    joblib.dump(clf_model, model_path)
    print("مدل نمونه ساخته و در پوشه models ذخیره شد.")

### ۴.۱. تست و ارزیابی مدل طبقه‌بندی

In [ ]:
# تعیین ویژگی‌های مورد نیاز مدل طبقه‌بندی بر اساس ساختار مدل شما
rf_features = ['Age', 'Gender_Encoded', 'Weight_kg', 'Height_cm', 'BMI', 'Body_Fat_Percent', 'Lean_Mass_kg', 'Activity_Score', 'Genetic_Score']

X_test_sample = df[rf_features].tail(50) # انتخاب بخشی از داده‌ها برای تست
y_true_sample = df['Protein_Class'].tail(50)

# پیش‌بینی
y_pred_clf = clf_model.predict(X_test_sample)

# نمایش گزارش ارزیابی
print("=== گزارش ارزیابی مدل Random Forest ===")
print(f"Accuracy: {accuracy_score(y_true_sample, y_pred_clf):.4f}\n")
print(classification_report(y_true_sample, y_pred_clf))

## ۵. ذخیره‌سازی نهایی مدل رگرسیون
برای اینکه مدل رگرسیون خطی چندگانه را هم مانند رندوم فارست در پوشه `models` داشته باشید، آن را ذخیره می‌کنیم.

In [ ]:
reg_model_path = os.path.join(models_dir, "multiple_linear_regression_model.pkl")
joblib.dump(mlR_model, reg_model_path)
print(f"مدل رگرسیون با موفقیت در مسیر زیر ذخیره شد:\n{reg_model_path}")